# Holosoma G1 FastSAC Slope Robustness

This notebook evaluates a trained `exp:g1-29dof-fast-sac` policy on a controlled grid of planar ground gradients.

## Executive Summary

IsaacGym was the representative path for this run. MJWarp failed earlier on the default mixed-terrain setup with numerical instability, while IsaacGym ran the documented G1 FastSAC locomotion experiment stably under WSL2.

The evaluated checkpoint is a default mixed-terrain locomotion policy, not a flat-terrain workaround and not one of the OMOMO/LAFAN whole-body-tracking demos. The policy is good by the repo's listed tracking targets at the point used for evaluation: latest local training metrics were `rew_tracking_lin_vel=1.0139` and `rew_tracking_ang_vel=0.8756`.

The weakest tested uphill direction was 0.0° with max passing grade 0.05; the strongest was 112.5° with max passing grade 0.25.

## Training Setup

- Model: `/home/smp/projects/holosoma/logs/hv-g1-manager/20260508_090804-rtx3080_1024bs2048_isaacgym_fastsac_defaultmix_resume20k_to_50000-locomotion/model_0050000.onnx`
- Simulator: IsaacGym under WSL2
- Training baseline: documented G1 FastSAC locomotion experiment on default mixed terrain
- Hardware-adjusted parameters: `num_envs=1024`, `batch_size=2048`, `compile=False`
- Training budget: resumed from 20k and completed the default 50k iteration target
- Evaluated checkpoint: final 50k ONNX export
- Latest parsed training iteration: `50000.0000`
- Latest parsed mean episode length: `999.2100`

The hardware adjustment keeps the same experiment, reward, robot, command, and mixed terrain setup. It reduces parallel environments and global SAC batch size to fit an RTX 3080 10 GiB card without OOM.

## Slope Evaluation Design

- Uphill direction `0°` means the commanded forward walk is directly uphill.
- `180°` means commanded forward walk is downhill.
- `90°` and `270°` are side-slope cases.
- Grade is rise/run; e.g. `0.20` grade is `11.3°`.
- Eval command: `{'x_mps': 0.5, 'y_mps': 0.0, 'yaw_rps': 0.0}`
- Eval duration: `8.0 s`, warmup: `1.0 s`
- Cells: `144`, env trials: `144`

## Success Rule

A trial is successful only if it avoids reset/fall, keeps mean forward velocity above
`0.25` m/s, progresses at least
`2.0` m, and keeps mean absolute yaw rate below
`0.6` rad/s.

Overall trial success rate: `0.472`.

## Training Curves

![Training metrics](figures/training_metrics.png)

## Directional Slope Results

![Slope success heatmap](figures/slope_success_heatmap.png)

![Max grade polar chart](figures/max_grade_polar.png)

| Uphill direction deg | Max passing grade | Slope angle deg |
|---:|---:|---:|
| 0.0 | 0.05 | 2.9 |
| 22.5 | 0.05 | 2.9 |
| 45.0 | 0.05 | 2.9 |
| 67.5 | 0.10 | 5.7 |
| 90.0 | 0.20 | 11.3 |
| 112.5 | 0.25 | 14.0 |
| 135.0 | 0.25 | 14.0 |
| 157.5 | 0.20 | 11.3 |
| 180.0 | 0.25 | 14.0 |
| 202.5 | 0.15 | 8.5 |
| 225.0 | 0.20 | 11.3 |
| 247.5 | 0.25 | 14.0 |
| 270.0 | 0.20 | 11.3 |
| 292.5 | 0.20 | 11.3 |
| 315.0 | 0.15 | 8.5 |
| 337.5 | 0.10 | 5.7 |

## Interpretation

The strongest sectors are side/rear-biased slopes, topping out at grade `0.25` in this run. The weakest sectors are forward and forward-diagonal uphill slopes, where the policy often only passes flat or very shallow grades. This is consistent with the default locomotion terrain mix emphasizing flat, rough, and low-obstacle terrain rather than a curriculum of continuous directional slopes.

## Limitations

- Each grade/direction cell is one deterministic trial. This isolates slope geometry, but it does not estimate stochastic robustness over spawn offsets or random pushes.
- The evaluation uses IsaacGym physics plus `holosoma_inference` ONNX observation/action logic; it is not the interactive SDK loop from `run_policy.py`.
- The OMOMO/LAFAN scripts exercise whole-body tracking policies, so they are not the right happy path for this locomotion slope robustness question.
- The final report uses the completed 50k run; earlier intermediate checkpoints were not used for final robustness evaluation.

## Artifacts

- Cell summary: `eval/slope_cell_summary.csv`
- Per-env trials: `eval/slope_env_trials.csv`
- Direction thresholds: `eval/slope_direction_thresholds.csv`
- Evaluation summary: `eval/slope_eval_summary.json`

## Additional Terrain Stress Tests

These tests add terrain families that are useful for debugging beyond planar slopes: rough sinusoidal terrain, uphill/downhill stairs, square pits, raised blocks, and a cross-gap. They are not a substitute for the default mixed-terrain training metric; they expose likely deployment failure modes.

Overall terrain-stress trial success rate: `0.722`.

![Additional terrain stress tests](figures/terrain_stress_tests.png)

| Terrain type | Max passing magnitude | Unit | Passing cells | Tested cells |
|---|---:|---|---:|---:|
| blocks | 0.120 | m_height | 5 | 6 |
| cross_gap | 0.150 | m_width | 3 | 6 |
| pits | 0.100 | m_depth | 5 | 6 |
| rough | 0.100 | m_amp | 6 | 6 |
| stairs_down | 0.100 | m_step | 5 | 6 |
| stairs_up | 0.040 | m_step | 2 | 6 |

- Stress cell summary: `eval/terrain_stress_cell_summary.csv`
- Stress thresholds: `eval/terrain_stress_thresholds.csv`
- Stress evaluation summary: `eval/terrain_stress_eval_summary.json`



In [ ]:
from pathlib import Path
import csv
import math
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name != "slope_robustness_50k":
    ROOT = ROOT / "reports" / "slope_robustness_50k"

cell_summary = ROOT / "eval/slope_cell_summary.csv"
direction_thresholds = ROOT / "eval/slope_direction_thresholds.csv"
env_trials = ROOT / "eval/slope_env_trials.csv"

with cell_summary.open(newline="") as f:
    cells = list(csv.DictReader(f))
with direction_thresholds.open(newline="") as f:
    thresholds = list(csv.DictReader(f))
with env_trials.open(newline="") as f:
    trials = list(csv.DictReader(f))

print(f"Loaded {len(cells)} slope cells and {len(trials)} trials")
print("Worst directions by max successful grade:")
for row in sorted(thresholds, key=lambda r: float(r["max_success_grade"]))[:5]:
    print(f"  {float(row['direction_deg']):6.1f} deg: grade={float(row['max_success_grade']):.2f}, angle={float(row['max_success_slope_angle_deg']):.1f} deg")


In [ ]:
print("Direction threshold table")
print("deg   max_grade   slope_angle_deg")
for row in sorted(thresholds, key=lambda r: float(r["direction_deg"])):
    print(f"{float(row['direction_deg']):5.1f} {float(row['max_success_grade']):10.2f} {float(row['max_success_slope_angle_deg']):15.1f}")


In [ ]:
stress_cell_summary = ROOT / "eval/terrain_stress_cell_summary.csv"
stress_thresholds = ROOT / "eval/terrain_stress_thresholds.csv"
with stress_cell_summary.open(newline="") as f:
    stress_cells = list(csv.DictReader(f))
with stress_thresholds.open(newline="") as f:
    stress_thr = list(csv.DictReader(f))
print("Additional terrain thresholds")
print("terrain_type     max_magnitude   unit")
for row in sorted(stress_thr, key=lambda r: r["terrain_type"]):
    print(f"{row['terrain_type']:<16} {float(row['max_success_magnitude']):12.3f}   {row['unit']}")
